# **Tech Challenge - Grupo 24: Análise de Retenção de Clientes (Olist)**

Este notebook foca na análise de **Retenção e Fidelidade de Clientes**, utilizando a base consolidada do projeto.

**Responsável:** Allan Din'iz

### **Objetivos:**
*   Calcular Taxa de Recompra e Churn Rate.
*   Identificar fatores que impactam a fidelização.
*   Propor estratégias de ativação de clientes inativos.

## **1. Importação de Bibliotecas e Configurações**

In [1]:
#%pip install seaborn
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import warnings

warnings.filterwarnings('ignore')

# Configurações Visuais
plt.rcParams.update({"figure.facecolor": "#FFFFFF", "axes.facecolor": "#F8F8F8", "font.family": "sans-serif"})
AZUL, LARANJA, VERDE, CINZA = "#2c3e50", "#ff7f0e", "#27ae60", "#B4B2A9"

## **2. Carregamento dos Dados Originais**

Buscando as bases com os dados na nova pasta Datasets.

In [3]:
# Atualizado para buscar os dados corretamente na pasta Datasets
path_prefix = 'Datasets/'

try:
    df_clientes = pd.read_csv(os.path.join(path_prefix, 'olist_customers_dataset.csv'))
    df_pedidos = pd.read_csv(os.path.join(path_prefix, 'olist_orders_dataset.csv'))
    df_itens_pedido = pd.read_csv(os.path.join(path_prefix, 'olist_order_items_dataset.csv'))
    df_pagamentos = pd.read_csv(os.path.join(path_prefix, 'olist_order_payments_dataset.csv'))
    df_avaliacoes = pd.read_csv(os.path.join(path_prefix, 'olist_order_reviews_dataset.csv'))
    df_produtos = pd.read_csv(os.path.join(path_prefix, 'olist_products_dataset.csv'))
    df_vendedores = pd.read_csv(os.path.join(path_prefix, 'olist_sellers_dataset.csv'))
    df_traducao_categorias = pd.read_csv(os.path.join(path_prefix, 'product_category_name_translation.csv'))
    print("✅ Todas as bases foram carregadas com sucesso!")
except Exception as e:
    print(f"❌ Erro ao carregar as bases: {e}")

✅ Todas as bases foram carregadas com sucesso!


## **3. Tratamento de Dados (Conforme Grupo)**

In [4]:
# 1. Conversão de Datas
colunas_datas = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in colunas_datas: df_pedidos[col] = pd.to_datetime(df_pedidos[col], errors='coerce')

# 2. Padronização de IDs para o Merge
for df in [df_pedidos, df_itens_pedido, df_pagamentos, df_avaliacoes]:
    if 'order_id' in df.columns: df['order_id'] = df['order_id'].astype(str)
    
# 3. Limpeza de Clientes
df_clientes['customer_zip_code_prefix'] = df_clientes['customer_zip_code_prefix'].astype(str).str.zfill(5)
df_clientes['customer_city'] = df_clientes['customer_city'].str.title()
print("✅ Tratamento concluído.")

✅ Tratamento concluído.


## **4. Criação da Base Consolidada Oficial**

In [5]:
# Agrupando Itens (Preço e Frete total)
df_itens_agrupado = df_itens_pedido.groupby('order_id').agg({'price': 'sum', 'freight_value': 'sum', 'product_id': 'first', 'seller_id': 'first'}).reset_index()

# Agrupando Pagamentos
df_pagamentos_agrupado = df_pagamentos.groupby('order_id').agg({'payment_value': 'sum', 'payment_type': lambda x: '/'.join(x.unique().astype(str)), 'payment_installments': 'max'}).reset_index()

# Agrupando Avaliações
df_avaliacoes_agrupado = df_avaliacoes.groupby('order_id').agg({'review_score': 'mean', 'review_id': 'first'}).reset_index()

# O MERGE CENTRAL
df_consolidado = pd.merge(df_pedidos, df_itens_agrupado, on='order_id', how='left')
df_consolidado = pd.merge(df_consolidado, df_pagamentos_agrupado, on='order_id', how='left')
df_consolidado = pd.merge(df_consolidado, df_avaliacoes_agrupado, on='order_id', how='left')
df_consolidado = pd.merge(df_consolidado, df_produtos, on='product_id', how='left')
df_consolidado = pd.merge(df_consolidado, df_clientes, on='customer_id', how='left')
df_consolidado = pd.merge(df_consolidado, df_vendedores, on='seller_id', how='left')

df_consolidado["atrasado"] = df_consolidado["order_delivered_customer_date"] > df_consolidado["order_estimated_delivery_date"]
print(f"✅ df_consolidado criada com {df_consolidado.shape[0]} linhas.")

✅ df_consolidado criada com 99441 linhas.


## **7. Sugestões de Melhoria**

Para um retorno efetivo, propomos trabalhar em **duas frentes** (curto e longo prazo):

1.  **Programa de Pontos**: Criar sistema de recompensa para converter a 2ª compra.
2.  **Lembrete de 'Sumiço'**: Automação para clientes inativos há mais de 4 meses.
3.  **Cuidado com Atrasos**: Cupom de desculpas imediato para problemas de entrega.
4.  **Sugestões do seu Jeito**: Cross-selling baseando na primeira compra.
5.  **Ajuste da Operação (Problema Raiz)**: Negociar novos contratos com transportadoras. Como a logística não acompanhou as vendas, essa parte leva mais tempo (meses de trabalho), mas é a solução real a longo prazo.